In [ ]:
# =============================================================================
# PACE-VCF Comprehensive Metrics Validation (UPDATED for v2)
# =============================================================================
# Purpose: Validate all PACE metrics files:
#   - MODIS_Metrics.tif (MODIS-compatible metrics) - NOW ~304 bands
#   - PACE_Metrics.tif (PACE hyperspectral indices) - ~146 bands
#   - PACE_AltSort_Metrics.tif (Alternative sorting metrics) - ~246 bands
#
# UPDATES for v2:
#   - Updated expected band counts
#   - Added Band31 (thermal) metrics validation
#   - Added Snow metrics validation
#   - Added QA metrics validation (diagnostic only)
#   - Updated value ranges for thermal difference metrics
#
# Author: MJ Frost
# Date: August 2026
# =============================================================================

# %% Cell 1: Configuration
import numpy as np
from pathlib import Path
from osgeo import gdal, osr
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Tuple, Optional
import warnings

warnings.filterwarnings('ignore')
gdal.UseExceptions()

# =============================================================================
# CONFIGURATION
# =============================================================================

PACE_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/output")
MODIS_BASE = Path("/explore/nobackup/projects/ilab/projects/MODIS-VCF/processedTiles/MOD44C")

# Years
PACE_YEAR = 2025
MODIS_YEAR = 2019

# Tiles to validate
TILES = ['h08v04', 'h09v05', 'h12v09', 'h16v01', 'h20v06', 'h31v11']
TILE_NAMES = {
    'h08v04': 'N. America (West)',
    'h09v05': 'N. America (Central)',
    'h12v09': 'Amazon',
    'h16v01': 'Europe/Iceland',
    'h20v06': 'Sahel/Africa',
    'h31v11': 'Australia'
}

# =============================================================================
# UPDATED: Expected file structure for v2 metrics
# =============================================================================
EXPECTED_FILES = {
    'MODIS_Metrics.tif': {'min_bands': 290, 'max_bands': 320},   # Was 250-280, now includes Band31, Snow, QA
    'PACE_Metrics.tif': {'min_bands': 140, 'max_bands': 160},    # ~146 bands
    'PACE_AltSort_Metrics.tif': {'min_bands': 230, 'max_bands': 260}  # ~246 bands
}

# Grid sizes
PACE_TILE_SIZE = 600
MODIS_TILE_SIZE = 4800

# No-data value
NO_DATA = -10001

# Sinusoidal projection parameters
MODIS_SPHERE_RADIUS = 6371007.181
MODIS_TILE_SIZE_M = 1111950.5196666666
MODIS_UPPER_LEFT_X = -20015109.354
MODIS_UPPER_LEFT_Y = 10007554.677

print("="*80)
print("PACE-VCF COMPREHENSIVE METRICS VALIDATION (v2)")
print("="*80)
print(f"Tiles to validate: {len(TILES)}")
print(f"PACE Year: {PACE_YEAR}")
print(f"MODIS Year (for comparison): {MODIS_YEAR}")
print()
print("v2 Updates:")
print("  - Band31 (thermal) basic metrics")
print("  - Snow detection and snow-free metrics")
print("  - QA observation count metrics")


# %% Cell 2: Helper Functions

def get_expected_geotransform(tile: str) -> tuple:
    """Calculate expected geotransform for a tile at PACE resolution."""
    h = int(tile[1:3])
    v = int(tile[4:6])
    
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    pixel_size = MODIS_TILE_SIZE_M / PACE_TILE_SIZE
    
    return (min_x, pixel_size, 0, max_y, 0, -pixel_size)


def check_projection(ds: gdal.Dataset) -> Tuple[bool, str]:
    """Check if dataset has correct sinusoidal projection."""
    proj = ds.GetProjection()
    
    if not proj:
        return False, "No projection defined"
    
    if 'sinu' in proj.lower() or 'sinusoidal' in proj.lower():
        return True, "MODIS Sinusoidal"
    else:
        return False, f"Unexpected projection"


def check_geotransform(ds: gdal.Dataset, tile: str) -> Tuple[bool, str]:
    """Check if geotransform matches expected values for tile."""
    gt = ds.GetGeoTransform()
    expected_gt = get_expected_geotransform(tile)
    
    tol = 1.0  # 1 meter tolerance
    
    if abs(gt[0] - expected_gt[0]) > tol or abs(gt[3] - expected_gt[3]) > tol:
        return False, f"Origin mismatch: got ({gt[0]:.0f}, {gt[3]:.0f}), expected ({expected_gt[0]:.0f}, {expected_gt[3]:.0f})"
    
    if abs(gt[1] - expected_gt[1]) > 1.0:
        return False, f"Pixel size X mismatch: got {gt[1]:.2f}, expected {expected_gt[1]:.2f}"
    
    return True, "OK"


def get_band_stats(ds: gdal.Dataset, band_idx: int) -> Dict:
    """Get statistics for a single band."""
    band = ds.GetRasterBand(band_idx)
    data = band.ReadAsArray()
    name = band.GetDescription() or f"Band_{band_idx}"
    
    valid = data[data != NO_DATA]
    total = data.size
    
    if len(valid) > 0:
        return {
            'name': name,
            'min': float(valid.min()),
            'max': float(valid.max()),
            'mean': float(valid.mean()),
            'std': float(valid.std()),
            'valid_count': len(valid),
            'valid_pct': 100 * len(valid) / total,
            'total': total
        }
    else:
        return {
            'name': name,
            'min': np.nan, 'max': np.nan, 'mean': np.nan, 'std': np.nan,
            'valid_count': 0, 'valid_pct': 0, 'total': total
        }


def load_modis_metric(tile: str, metric_file: str, band_idx: int) -> Optional[np.ndarray]:
    """Load MODIS metric and resample to PACE resolution."""
    filepath = MODIS_BASE / tile / str(MODIS_YEAR) / "3-Metrics" / metric_file
    
    if not filepath.exists():
        return None
    
    ds = gdal.Open(str(filepath))
    if ds is None:
        return None
    
    data = ds.GetRasterBand(band_idx + 1).ReadAsArray().astype(np.float32)
    ds = None
    
    data[data == NO_DATA] = np.nan
    
    if data.shape == (MODIS_TILE_SIZE, MODIS_TILE_SIZE):
        reshaped = data.reshape(PACE_TILE_SIZE, 8, PACE_TILE_SIZE, 8)
        with np.errstate(all='ignore'):
            data = np.nanmean(reshaped, axis=(1, 3))
    
    return data


def load_pace_band(tile: str, filename: str, band_name: str) -> Optional[np.ndarray]:
    """Load a specific band from a PACE metrics file."""
    filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / filename
    
    if not filepath.exists():
        return None
    
    ds = gdal.Open(str(filepath))
    if ds is None:
        return None
    
    for i in range(1, ds.RasterCount + 1):
        band = ds.GetRasterBand(i)
        if band.GetDescription() == band_name:
            data = band.ReadAsArray().astype(np.float32)
            data[data == NO_DATA] = np.nan
            ds = None
            return data
    
    ds = None
    return None


print("Helper functions defined")


# %% Cell 3: Section 1 - File Existence and Structure

print("\n" + "="*80)
print("SECTION 1: FILE EXISTENCE AND STRUCTURE")
print("="*80)

file_status = {}

for tile in TILES:
    tile_name = TILE_NAMES.get(tile, tile)
    print(f"\n{tile} ({tile_name}):")
    print("-"*60)
    
    file_status[tile] = {}
    
    for filename, expected in EXPECTED_FILES.items():
        filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / filename
        
        if filepath.exists():
            ds = gdal.Open(str(filepath))
            n_bands = ds.RasterCount
            x_size = ds.RasterXSize
            y_size = ds.RasterYSize
            dtype = gdal.GetDataTypeName(ds.GetRasterBand(1).DataType)
            
            band_ok = expected['min_bands'] <= n_bands <= expected['max_bands']
            band_status = "✓" if band_ok else "⚠️"
            
            dim_ok = x_size == PACE_TILE_SIZE and y_size == PACE_TILE_SIZE
            dim_status = "✓" if dim_ok else "✗"
            
            print(f"  {filename}:")
            print(f"    Bands: {n_bands} {band_status} (expected {expected['min_bands']}-{expected['max_bands']})")
            print(f"    Dimensions: {x_size}×{y_size} {dim_status}")
            print(f"    Data type: {dtype}")
            
            file_status[tile][filename] = {
                'exists': True,
                'bands': n_bands,
                'dimensions': (x_size, y_size),
                'dtype': dtype,
                'band_ok': band_ok,
                'dim_ok': dim_ok
            }
            
            ds = None
        else:
            print(f"  {filename}: ✗ NOT FOUND")
            file_status[tile][filename] = {'exists': False}

# Summary table
print("\n" + "-"*80)
print("FILE EXISTENCE SUMMARY")
print("-"*80)
print(f"{'Tile':<12}", end='')
for filename in EXPECTED_FILES.keys():
    short_name = filename.replace('_Metrics.tif', '')
    print(f"{short_name:>18}", end='')
print()
print("-"*80)

for tile in TILES:
    print(f"{tile:<12}", end='')
    for filename in EXPECTED_FILES.keys():
        status = file_status[tile].get(filename, {})
        if status.get('exists'):
            n_bands = status.get('bands', 0)
            print(f"{n_bands:>12} bands ✓", end='')
        else:
            print(f"{'MISSING':>18}", end='')
    print()


# %% Cell 4: Section 2 - Projection and Geotransform Validation
# [KEEP AS-IS - no changes needed]

print("\n" + "="*80)
print("SECTION 2: PROJECTION AND GEOTRANSFORM VALIDATION")
print("="*80)

geo_status = {}

for tile in TILES:
    tile_name = TILE_NAMES.get(tile, tile)
    print(f"\n{tile} ({tile_name}):")
    
    geo_status[tile] = {}
    
    filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "MODIS_Metrics.tif"
    
    if not filepath.exists():
        print("  File not found, skipping...")
        continue
    
    ds = gdal.Open(str(filepath))
    
    proj_ok, proj_msg = check_projection(ds)
    print(f"  Projection: {proj_msg} {'✓' if proj_ok else '✗'}")
    
    gt_ok, gt_msg = check_geotransform(ds, tile)
    print(f"  Geotransform: {gt_msg} {'✓' if gt_ok else '✗'}")
    
    gt = ds.GetGeoTransform()
    print(f"    Origin: ({gt[0]:.0f}, {gt[3]:.0f})")
    print(f"    Pixel size: {gt[1]:.2f} m ({gt[1]/1000:.2f} km)")
    
    geo_status[tile] = {
        'proj_ok': proj_ok,
        'gt_ok': gt_ok,
        'origin': (gt[0], gt[3]),
        'pixel_size': gt[1]
    }
    
    ds = None


# %% Cell 5: Section 3 - Band Inventory (UPDATED)

print("\n" + "="*80)
print("SECTION 3: BAND INVENTORY BY FILE TYPE")
print("="*80)

sample_tile = TILES[0]

for filename in EXPECTED_FILES.keys():
    filepath = PACE_BASE / sample_tile / str(PACE_YEAR) / "3-Metrics" / filename
    
    if not filepath.exists():
        print(f"\n{filename}: NOT FOUND")
        continue
    
    ds = gdal.Open(str(filepath))
    bands = []
    
    for i in range(1, ds.RasterCount + 1):
        band = ds.GetRasterBand(i)
        bands.append(band.GetDescription() or f"Band_{i}")
    
    ds = None
    
    print(f"\n{filename}: {len(bands)} bands")
    print("-"*60)
    
    # UPDATED: Categorize bands including new v2 categories
    categories = {}
    
    for band_name in bands:
        if band_name.startswith('QA_'):
            cat = 'QA Observation Counts (NEW)'
        elif 'Snow' in band_name:
            cat = 'Snow Metrics (NEW)'
        elif 'Band31' in band_name:
            cat = 'Band31 Thermal (NEW)'
        elif 'UnsortedMonthly' in band_name:
            cat = 'Unsorted Monthly'
        elif 'Brownest' in band_name:
            cat = 'Brownest (Senescence)'
        elif 'Anthocyanin' in band_name:
            cat = 'Anthocyanin-Sorted (mARI)'
        elif 'Stressed' in band_name or 'AmpStress' in band_name:
            cat = 'Stress-Sorted (PRI)'
        elif 'Chlorophyll' in band_name or 'Carotenoid' in band_name or 'PigmentRatio' in band_name:
            cat = 'Pigment-Sorted (CCI)'
        elif 'Phenology' in band_name:
            cat = 'Phenology'
        elif 'CrossIndex' in band_name:
            cat = 'Cross-Index'
        elif 'PACEIndex' in band_name:
            cat = 'PACE Indices'
        elif 'Greenest' in band_name or 'Greenness' in band_name:
            cat = 'Greenness-Sorted'
        elif 'Warmest' in band_name or 'Coolest' in band_name:
            cat = 'Temperature-Sorted'
        elif 'Temp' in band_name and 'Band' not in band_name:
            cat = 'Temperature Metrics'
        elif 'Lowest' in band_name:
            cat = 'Lowest-N'
        elif any(x in band_name for x in ['Min', 'Max', 'Median', 'Amp', 'Mean']):
            cat = 'Basic Statistics'
        else:
            cat = 'Other'
        
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(band_name)
    
    # Sort to show NEW categories first
    sorted_cats = sorted(categories.items(), key=lambda x: (0 if 'NEW' in x[0] else 1, x[0]))
    
    for cat, cat_bands in sorted_cats:
        marker = "🆕" if 'NEW' in cat else "  "
        print(f"{marker} {cat}: {len(cat_bands)} bands")
        for b in cat_bands[:3]:
            print(f"      - {b}")
        if len(cat_bands) > 3:
            print(f"      ... and {len(cat_bands) - 3} more")


# %% Cell 6: Section 4 - Valid Pixel Coverage (UPDATED)

print("\n" + "="*80)
print("SECTION 4: VALID PIXEL COVERAGE")
print("="*80)

# UPDATED: Sample bands to check coverage including v2 metrics
sample_bands = {
    'MODIS_Metrics.tif': [
        'BandReflMax-NDVI',
        'BandReflMedian-Band_1',
        'BandReflMedian-Band_6',
        'TempMeanWarmest3',
        # NEW v2 metrics
        'BandReflMax-Band31',
        'BandReflMean-Band31',
        'AmpBandRefl-Band31',
        'ThermalGreenBrownDiff-Band31',
        'Snow-CompositeCount',
        'SnowFree-NDVI-Max',
        'QA_ObsCount-Optical-Total',
        'QA_ObsCount-Thermal-Total',
    ],
    'PACE_Metrics.tif': [
        'PACEIndex-EVI-Median',
        'PACEIndex-PRI-Median',
        'PACEIndex-mARI-Median',
        'PACEIndex-REP-Median',
        'PACEIndex-REIP-Median',
    ],
    'PACE_AltSort_Metrics.tif': [
        'PACEIndex-EVI-Brownest3Mean',
        'Phenology-GrowingSeasonLength',
        'Phenology-MaxGreenupRate',
        'CrossIndex-mARI-SeasonalRange',
        'NDVI-AtGreenUp',
        'NDVI-Brownest',              # <-- Changed from 'NDVI-AtMinNDVI'
        'PACEIndex-EVI-AtMinNDVI',    # <-- Add this if you want to check index at min NDVI
    ]
}

coverage_results = {}

for tile in TILES:
    tile_name = TILE_NAMES.get(tile, tile)
    print(f"\n{tile} ({tile_name}):")
    print("-"*60)
    
    coverage_results[tile] = {}
    
    for filename, bands_to_check in sample_bands.items():
        filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / filename
        
        if not filepath.exists():
            print(f"  {filename}: NOT FOUND")
            continue
        
        ds = gdal.Open(str(filepath))
        
        print(f"  {filename}:")
        
        for band_name in bands_to_check:
            found = False
            for i in range(1, ds.RasterCount + 1):
                band = ds.GetRasterBand(i)
                if band.GetDescription() == band_name:
                    data = band.ReadAsArray()
                    valid = np.sum(data != NO_DATA)
                    total = data.size
                    pct = 100 * valid / total
                    
                    # Flag if coverage is unexpectedly low
                    flag = ""
                    if pct < 50 and not band_name.startswith('Snow') and not band_name.startswith('QA_'):
                        flag = " ⚠️"
                    elif pct < 1:
                        flag = " ❌"
                    
                    # Mark new v2 metrics
                    marker = "🆕" if any(x in band_name for x in ['Band31', 'Snow', 'QA_']) else "  "
                    
                    print(f"  {marker} {band_name}: {pct:.1f}% valid ({valid:,}/{total:,}){flag}")
                    
                    coverage_results[tile][band_name] = pct
                    found = True
                    break
            
            if not found:
                print(f"    {band_name}: NOT FOUND")
        
        ds = None


# %% Cell 7: Section 5 - Value Range Validation (UPDATED)

print("\n" + "="*80)
print("SECTION 5: VALUE RANGE VALIDATION")
print("="*80)

# UPDATED: Expected ranges including new metric types
expected_ranges = {
    'NDVI': {'min': -1000, 'max': 1000, 'typical_mean': (0, 800)},
    'Reflectance': {'min': 0, 'max': 10000, 'typical_mean': (100, 5000)},
    'Temperature': {'min': 25000, 'max': 35000, 'typical_mean': (28000, 32000)},
    'ThermalDiff': {'min': -5000, 'max': 10000, 'typical_mean': (0, 5000)},  # NEW: For amplitude/diff
    'Index': {'min': -10000, 'max': 10000, 'typical_mean': (-5000, 5000)},
    'Phenology_Count': {'min': 0, 'max': 12, 'typical_mean': (1, 11)},
    'Phenology_Time': {'min': 0, 'max': 11, 'typical_mean': (0, 11)},
    'Snow_Count': {'min': 0, 'max': 12, 'typical_mean': (0, 6)},  # NEW
    'QA_Count': {'min': 0, 'max': 12, 'typical_mean': (1, 12)},   # NEW
    'REP': {'min': 6800, 'max': 7600, 'typical_mean': (7000, 7200)},  # NEW: Red Edge Position (×10)
}

def classify_metric_type(band_name: str) -> str:
    """Classify metric type for range checking."""
    if band_name.startswith('QA_'):
        return 'QA_Count'
    elif 'Snow-CompositeCount' in band_name or 'Snow-FreeCompositeCount' in band_name:
        return 'Snow_Count'
    elif 'NDVI' in band_name and 'Index' not in band_name:
        return 'NDVI'
    elif 'REP' in band_name or 'REIP' in band_name:
        return 'REP'
    elif 'Amp' in band_name and 'Band31' in band_name:
        return 'ThermalDiff'
    elif 'ThermalGreenBrownDiff' in band_name:
        return 'ThermalDiff'
    elif 'Band31' in band_name:
        return 'Temperature'
    elif 'Temp' in band_name and 'Band' not in band_name:
        return 'Temperature'
    elif 'PACEIndex' in band_name or 'CrossIndex' in band_name:
        return 'Index'
    elif 'GrowingSeasonLength' in band_name or 'GSL' in band_name:
        return 'Phenology_Count'
    elif 'TimeTo' in band_name:
        return 'Phenology_Time'
    elif 'Band_' in band_name:
        return 'Reflectance'
    else:
        return 'Index'

# UPDATED: Sample bands for range checking including v2
range_check_bands = {
    'MODIS_Metrics.tif': [
        ('BandReflMax-NDVI', 'NDVI'),
        ('BandReflMedian-Band_1', 'Reflectance'),
        ('BandReflMedian-Band_6', 'Reflectance'),
        ('TempMeanWarmest3', 'Temperature'),
        # NEW v2
        ('BandReflMax-Band31', 'Temperature'),
        ('AmpBandRefl-Band31', 'ThermalDiff'),
        ('ThermalGreenBrownDiff-Band31', 'ThermalDiff'),
        ('Snow-CompositeCount', 'Snow_Count'),
        ('QA_ObsCount-Optical-Total', 'QA_Count'),
    ],
    'PACE_Metrics.tif': [
        ('PACEIndex-EVI-Median', 'Index'),
        ('PACEIndex-PRI-Median', 'Index'),
        ('PACEIndex-mARI-Median', 'Index'),
        ('PACEIndex-REP-Median', 'REP'),
    ],
    'PACE_AltSort_Metrics.tif': [
        ('Phenology-GrowingSeasonLength', 'Phenology_Count'),
        ('Phenology-TimeToPeak', 'Phenology_Time'),
        ('CrossIndex-mARI-SeasonalRange', 'Index'),
    ]
}

print(f"\n{'Tile':<10} {'Band':<45} {'Min':>10} {'Max':>10} {'Mean':>10} {'Status':<10}")
print("-"*100)

for tile in TILES[:]:
    for filename, bands in range_check_bands.items():
        filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / filename
        
        if not filepath.exists():
            continue
        
        ds = gdal.Open(str(filepath))
        
        for band_name, metric_type in bands:
            for i in range(1, ds.RasterCount + 1):
                band = ds.GetRasterBand(i)
                if band.GetDescription() == band_name:
                    data = band.ReadAsArray()
                    valid = data[data != NO_DATA]
                    
                    if len(valid) > 0:
                        min_val = valid.min()
                        max_val = valid.max()
                        mean_val = valid.mean()
                        
                        exp = expected_ranges.get(metric_type, {})
                        status = "✓"
                        
                        if exp:
                            if min_val < exp.get('min', -np.inf) * 1.5:
                                status = "⚠️ Low min"
                            elif max_val > exp.get('max', np.inf) * 1.5:
                                status = "⚠️ High max"
                            elif exp.get('typical_mean'):
                                tm = exp['typical_mean']
                                if mean_val < tm[0] * 0.5 or mean_val > tm[1] * 1.5:
                                    status = "⚠️ Unusual mean"
                        
                        # Mark new metrics
                        marker = "🆕" if any(x in band_name for x in ['Band31', 'Snow', 'QA_']) else ""
                        short_name = band_name[:43] + '..' if len(band_name) > 45 else band_name
                        print(f"{tile:<10} {marker}{short_name:<43} {min_val:>10.0f} {max_val:>10.0f} {mean_val:>10.1f} {status:<10}")
                    break
        
        ds = None


# %% Cell 8: Section 6 - MODIS Comparison
# [KEEP AS-IS - no changes needed for basic comparison]

print("\n" + "="*80)
print("SECTION 6: PACE vs MODIS COMPARISON (Matching Metrics)")
print("="*80)

comparison_metrics = [
    ('BandReflMax-NDVI', 'BandReflMax.tif', 7),
    ('BandReflMin-NDVI', 'BandReflMin.tif', 7),
    ('BandReflMedian-NDVI', 'BandReflMedian.tif', 7),
    ('BandReflMedian-Band_1', 'BandReflMedian.tif', 0),
    ('BandReflMedian-Band_2', 'BandReflMedian.tif', 1),
    ('BandReflMedian-Band_6', 'BandReflMedian.tif', 5),  # Added Band_6
    ('Greenest3MeanBandRefl-NDVI', 'Greenest3MeanBandRefl.tif', 7),
    ('TempMeanWarmest3', 'TempMeanWarmest3.tif', 0),
]

print(f"\n{'Metric':<35} {'Tile':<10} {'PACE Mean':>12} {'MODIS Mean':>12} {'Corr':>8} {'RMSE':>10}")
print("-"*95)

comparison_results = []

for pace_band, modis_file, modis_idx in comparison_metrics:
    for tile in TILES[:]:
        pace_data = load_pace_band(tile, 'MODIS_Metrics.tif', pace_band)
        modis_data = load_modis_metric(tile, modis_file, modis_idx)
        
        if pace_data is None or modis_data is None:
            continue
        
        valid = ~np.isnan(pace_data) & ~np.isnan(modis_data)
        
        if np.sum(valid) < 100:
            continue
        
        pace_valid = pace_data[valid]
        modis_valid = modis_data[valid]
        
        pace_mean = np.mean(pace_valid)
        modis_mean = np.mean(modis_valid)
        corr = np.corrcoef(pace_valid, modis_valid)[0, 1]
        rmse = np.sqrt(np.mean((pace_valid - modis_valid)**2))
        
        short_name = pace_band[:33] + '..' if len(pace_band) > 35 else pace_band
        print(f"{short_name:<35} {tile:<10} {pace_mean:>12.1f} {modis_mean:>12.1f} {corr:>8.3f} {rmse:>10.1f}")
        
        comparison_results.append({
            'metric': pace_band,
            'tile': tile,
            'pace_mean': pace_mean,
            'modis_mean': modis_mean,
            'correlation': corr,
            'rmse': rmse
        })

if comparison_results:
    df = pd.DataFrame(comparison_results)
    
    print("\n" + "-"*60)
    print("COMPARISON SUMMARY BY METRIC")
    print("-"*60)
    
    for metric in df['metric'].unique():
        metric_df = df[df['metric'] == metric]
        avg_corr = metric_df['correlation'].mean()
        avg_rmse = metric_df['rmse'].mean()
        
        status = "✓" if avg_corr > 0.7 else "⚠️" if avg_corr > 0.5 else "✗"
        short_name = metric[:45] + '..' if len(metric) > 47 else metric
        print(f"  {short_name:<47} r={avg_corr:.3f} {status}")


# %% Cell 9: Section 7 - Visual Comparison (UPDATED)

print("\n" + "="*80)
print("SECTION 7: VISUAL COMPARISON")
print("="*80)

def create_comparison_figure(tile: str, save_path: Path = None):
    """Create visual comparison figure for a tile - UPDATED for v2."""
    
    fig, axes = plt.subplots(5, 3, figsize=(16, 25))  # Added 5th row
    fig.suptitle(f'PACE-VCF Metrics Validation (v2): {tile} ({TILE_NAMES.get(tile, "")})', 
                 fontsize=16, fontweight='bold')
    
    # Row 1: NDVI comparison (PACE vs MODIS)
    pace_ndvi = load_pace_band(tile, 'MODIS_Metrics.tif', 'BandReflMax-NDVI')
    modis_ndvi = load_modis_metric(tile, 'BandReflMax.tif', 7)
    
    if pace_ndvi is not None:
        im = axes[0, 0].imshow(pace_ndvi, cmap='YlGn', vmin=0, vmax=1000)
        axes[0, 0].set_title(f'PACE Max NDVI ({PACE_YEAR})')
        plt.colorbar(im, ax=axes[0, 0], shrink=0.8)
    axes[0, 0].axis('off')
    
    if modis_ndvi is not None:
        im = axes[0, 1].imshow(modis_ndvi, cmap='YlGn', vmin=0, vmax=1000)
        axes[0, 1].set_title(f'MODIS Max NDVI ({MODIS_YEAR})')
        plt.colorbar(im, ax=axes[0, 1], shrink=0.8)
    axes[0, 1].axis('off')
    
    if pace_ndvi is not None and modis_ndvi is not None:
        diff = pace_ndvi - modis_ndvi
        vmax = np.nanpercentile(np.abs(diff), 95)
        im = axes[0, 2].imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[0, 2].set_title('Difference (PACE - MODIS)')
        plt.colorbar(im, ax=axes[0, 2], shrink=0.8)
    axes[0, 2].axis('off')
    
    # Row 2: PACE Indices
    evi = load_pace_band(tile, 'PACE_Metrics.tif', 'PACEIndex-EVI-Median')
    pri = load_pace_band(tile, 'PACE_Metrics.tif', 'PACEIndex-PRI-Median')
    mari = load_pace_band(tile, 'PACE_Metrics.tif', 'PACEIndex-mARI-Median')
    
    if evi is not None:
        im = axes[1, 0].imshow(evi, cmap='YlGn', vmin=-500, vmax=1000)
        axes[1, 0].set_title('PACE EVI Median')
        plt.colorbar(im, ax=axes[1, 0], shrink=0.8)
    axes[1, 0].axis('off')
    
    if pri is not None:
        im = axes[1, 1].imshow(pri, cmap='RdYlGn', vmin=-200, vmax=200)
        axes[1, 1].set_title('PACE PRI Median')
        plt.colorbar(im, ax=axes[1, 1], shrink=0.8)
    axes[1, 1].axis('off')
    
    if mari is not None:
        im = axes[1, 2].imshow(mari, cmap='RdPu', vmin=-500, vmax=2000)
        axes[1, 2].set_title('PACE mARI Median')
        plt.colorbar(im, ax=axes[1, 2], shrink=0.8)
    axes[1, 2].axis('off')
    
    # Row 3: Alternative Sorting Metrics
    brownest = load_pace_band(tile, 'PACE_AltSort_Metrics.tif', 'PACEIndex-EVI-Brownest3Mean')
    stressed = load_pace_band(tile, 'PACE_AltSort_Metrics.tif', 'PACEIndex-EVI-MostStressed3Mean')
    high_anth = load_pace_band(tile, 'PACE_AltSort_Metrics.tif', 'PACEIndex-EVI-HighAnthocyanin3Mean')
    
    if brownest is not None:
        im = axes[2, 0].imshow(brownest, cmap='YlOrBr', vmin=-500, vmax=1000)
        axes[2, 0].set_title('EVI at Brownest (Senescence)')
        plt.colorbar(im, ax=axes[2, 0], shrink=0.8)
    axes[2, 0].axis('off')
    
    if stressed is not None:
        im = axes[2, 1].imshow(stressed, cmap='YlOrRd', vmin=-500, vmax=1000)
        axes[2, 1].set_title('EVI at Most Stressed')
        plt.colorbar(im, ax=axes[2, 1], shrink=0.8)
    axes[2, 1].axis('off')
    
    if high_anth is not None:
        im = axes[2, 2].imshow(high_anth, cmap='RdPu', vmin=-500, vmax=1000)
        axes[2, 2].set_title('EVI at High Anthocyanin')
        plt.colorbar(im, ax=axes[2, 2], shrink=0.8)
    axes[2, 2].axis('off')
    
    # Row 4: Phenology Metrics
    greenup = load_pace_band(tile, 'PACE_AltSort_Metrics.tif', 'Phenology-MaxGreenupRate')
    season_len = load_pace_band(tile, 'PACE_AltSort_Metrics.tif', 'Phenology-GrowingSeasonLength')
    time_peak = load_pace_band(tile, 'PACE_AltSort_Metrics.tif', 'Phenology-TimeToPeak')
    
    if greenup is not None:
        im = axes[3, 0].imshow(greenup, cmap='Greens', vmin=0, vmax=300)
        axes[3, 0].set_title('Max Green-up Rate')
        plt.colorbar(im, ax=axes[3, 0], shrink=0.8)
    axes[3, 0].axis('off')
    
    if season_len is not None:
        im = axes[3, 1].imshow(season_len, cmap='YlGn', vmin=0, vmax=12)
        axes[3, 1].set_title('Growing Season Length')
        plt.colorbar(im, ax=axes[3, 1], shrink=0.8)
    axes[3, 1].axis('off')
    
    if time_peak is not None:
        im = axes[3, 2].imshow(time_peak, cmap='plasma', vmin=0, vmax=11)
        axes[3, 2].set_title('Time to Peak (Composite #)')
        plt.colorbar(im, ax=axes[3, 2], shrink=0.8)
    axes[3, 2].axis('off')
    
    # Row 5: NEW v2 Metrics (Band31, Snow)
    thermal_max = load_pace_band(tile, 'MODIS_Metrics.tif', 'BandReflMax-Band31')
    thermal_amp = load_pace_band(tile, 'MODIS_Metrics.tif', 'AmpBandRefl-Band31')
    snow_count = load_pace_band(tile, 'MODIS_Metrics.tif', 'Snow-CompositeCount')
    
    if thermal_max is not None:
        im = axes[4, 0].imshow(thermal_max, cmap='RdYlBu_r', vmin=28000, vmax=32000)
        axes[4, 0].set_title('🆕 Band31 Max LST (×100 K)')
        plt.colorbar(im, ax=axes[4, 0], shrink=0.8)
    axes[4, 0].axis('off')
    
    if thermal_amp is not None:
        im = axes[4, 1].imshow(thermal_amp, cmap='YlOrRd', vmin=0, vmax=5000)
        axes[4, 1].set_title('🆕 Band31 Amplitude (×100 K)')
        plt.colorbar(im, ax=axes[4, 1], shrink=0.8)
    axes[4, 1].axis('off')
    
    if snow_count is not None:
        im = axes[4, 2].imshow(snow_count, cmap='Blues', vmin=0, vmax=12)
        axes[4, 2].set_title('🆕 Snow Composite Count')
        plt.colorbar(im, ax=axes[4, 2], shrink=0.8)
    axes[4, 2].axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()


for tile in TILES[:]:
    output_path = PACE_BASE / f"validation_v2_{tile}.png"
    create_comparison_figure(tile, output_path)


# %% Cell 10: Section 8 - NEW v2 Metrics Summary

print("\n" + "="*80)
print("SECTION 8: NEW v2 METRICS VALIDATION")
print("="*80)

print("\n--- Band31 (Thermal) Metrics ---")
band31_metrics = [
    'BandReflMin-Band31', 'BandReflMax-Band31', 'BandReflMean-Band31', 'BandReflMedian-Band31',
    'AmpBandRefl-Band31', 'Warmest3MeanBandRefl-Band31', 'Coolest3MeanBandRefl-Band31',
    'Greenest3MeanBandRefl-Band31', 'ThermalAtPeakNDVI-Band31', 'ThermalGreenBrownDiff-Band31'
]

for tile in TILES[:2]:
    print(f"\n{tile}:")
    filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "MODIS_Metrics.tif"
    if not filepath.exists():
        continue
    
    ds = gdal.Open(str(filepath))
    for band_name in band31_metrics[:5]:  # Check first 5
        for i in range(1, ds.RasterCount + 1):
            band = ds.GetRasterBand(i)
            if band.GetDescription() == band_name:
                data = band.ReadAsArray()
                valid = data[data != NO_DATA]
                if len(valid) > 0:
                    print(f"  {band_name}: {100*len(valid)/data.size:.1f}% valid, range [{valid.min():.0f}, {valid.max():.0f}]")
                else:
                    print(f"  {band_name}: ❌ 0% valid")
                break
    ds = None

print("\n--- Snow Metrics ---")
snow_metrics = [
    'Snow-CompositeCount', 'Snow-FreeCompositeCount', 'Snow-MaxNDSI', 'Snow-AtBrownest',
    'SnowFree-NDVI-Max', 'SnowFree-LST-Max', 'LST-AtFirstSnowFree'
]

for tile in TILES[:2]:
    print(f"\n{tile}:")
    filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "MODIS_Metrics.tif"
    if not filepath.exists():
        continue
    
    ds = gdal.Open(str(filepath))
    for band_name in snow_metrics[:5]:
        for i in range(1, ds.RasterCount + 1):
            band = ds.GetRasterBand(i)
            if band.GetDescription() == band_name:
                data = band.ReadAsArray()
                valid = data[data != NO_DATA]
                if len(valid) > 0:
                    print(f"  {band_name}: {100*len(valid)/data.size:.1f}% valid, range [{valid.min():.0f}, {valid.max():.0f}]")
                else:
                    print(f"  {band_name}: ❌ 0% valid")
                break
    ds = None

print("\n--- QA Observation Metrics ---")
qa_metrics = ['QA_ObsCount-Optical-Total', 'QA_ObsCount-Thermal-Total', 'QA_ObsCount-OpticalMinusThermal']

for tile in TILES[:2]:
    print(f"\n{tile}:")
    filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "MODIS_Metrics.tif"
    if not filepath.exists():
        continue
    
    ds = gdal.Open(str(filepath))
    for band_name in qa_metrics:
        for i in range(1, ds.RasterCount + 1):
            band = ds.GetRasterBand(i)
            if band.GetDescription() == band_name:
                data = band.ReadAsArray()
                valid = data[data != NO_DATA]
                if len(valid) > 0:
                    print(f"  {band_name}: mean={valid.mean():.1f}, range [{valid.min()}, {valid.max()}]")
                break
    ds = None


# %% Cell 11: Final Summary (UPDATED)

print("\n" + "="*80)
print("SECTION 9: VALIDATION SUMMARY")
print("="*80)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                     VALIDATION SUMMARY (v2)                                 │
├─────────────────────────────────────────────────────────────────────────────┤
│  Tiles Validated: {len(TILES):<55}│
│  PACE Year: {PACE_YEAR:<60}│
│  MODIS Comparison Year: {MODIS_YEAR:<48}│
├─────────────────────────────────────────────────────────────────────────────┤
│  FILE STATUS:                                                               │
│    MODIS_Metrics.tif:        {sum(1 for t in TILES if file_status.get(t, {}).get('MODIS_Metrics.tif', {}).get('exists')):>2}/{len(TILES)} tiles (~304 bands)                │
│    PACE_Metrics.tif:         {sum(1 for t in TILES if file_status.get(t, {}).get('PACE_Metrics.tif', {}).get('exists')):>2}/{len(TILES)} tiles (~146 bands)                │
│    PACE_AltSort_Metrics.tif: {sum(1 for t in TILES if file_status.get(t, {}).get('PACE_AltSort_Metrics.tif', {}).get('exists')):>2}/{len(TILES)} tiles (~246 bands)                │
├─────────────────────────────────────────────────────────────────────────────┤
│  NEW IN v2:                                                                 │
│    🆕 Band31 (thermal) metrics: ~15 bands                                   │
│    🆕 Snow detection metrics: ~12 bands                                     │
│    🆕 QA observation metrics: ~7 bands                                      │
├─────────────────────────────────────────────────────────────────────────────┤
│  SCALING:                                                                   │
│    Reflectance: × 10,000 (0.1 = 1000)                                      │
│    NDVI/Indices: × 1,000 (0.5 = 500)                                       │
│    Temperature: × 100 (300K = 30000)                                       │
│    Thermal Diff: × 100 (30K diff = 3000)                                   │
│    REP/REIP: × 10 (705nm = 7050)                                           │
│    No-data: -10001                                                          │
└─────────────────────────────────────────────────────────────────────────────┘
""")

all_files_present = all(
    file_status.get(t, {}).get(f, {}).get('exists', False)
    for t in TILES
    for f in EXPECTED_FILES.keys()
)

if all_files_present:
    print("✓ All expected files present for all tiles")
else:
    print("⚠️ Some files missing - check Section 1 for details")

if comparison_results:
    avg_corr = np.mean([r['correlation'] for r in comparison_results])
    if avg_corr > 0.7:
        print(f"✓ PACE-MODIS correlation is good (avg r = {avg_corr:.3f})")
    elif avg_corr > 0.5:
        print(f"⚠️ PACE-MODIS correlation is moderate (avg r = {avg_corr:.3f})")
    else:
        print(f"✗ PACE-MODIS correlation is low (avg r = {avg_corr:.3f})")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)
